# 01 - EDA: Data Slayer 4.0 (Exam Cheating Detection)

Task: binary video-clip classification, `0 = normal`, `1 = cheat`. Metric: **F1-macro**. 3 submissions/day.

Rules that shape the approach: no external data, no manual prediction. Pretrained weights are used as model init.

All heavy logic lives in `src/eda/explore.py`. Data is streamed: videos frame by frame, test JPEGs in small GPU
batches, so RAM stays low. **Do not publish this notebook with outputs: frames contain faces (competition privacy rule).**

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from dataset.labels import BEHAVIORS
from eda import explore as eda

DATA = Path.cwd().parent / "data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.width", 160)

## 1. Train videos

In [ ]:
vt = eda.video_table(DATA)
print(vt.shape, "| subjects:", vt.subject.nunique())
display(pd.crosstab(vt.behavior, [vt.label, vt.view]))
display(vt.groupby("view")[["fps", "n_frames", "duration", "width", "height"]].describe().T.round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for b, g in vt.groupby("behavior"):
    ax[0].hist(g.duration, bins=15, alpha=0.5, label=b)
ax[0].set_title("Video duration (s) per behavior")
ax[0].legend(fontsize=6)
vt.groupby("label").duration.sum().plot.bar(ax=ax[1], title="Total seconds per label (0=normal, 1=cheat)")
plt.tight_layout()

Findings:
- 26 subjects x 8 behaviors x 2 views = 416 videos, perfectly balanced; ~16 s each, ~6.7k one-second clips in total.
- Front camera: HEVC Main10 **HLG/BT.2020 (iPhone HDR)**; side camera: 8-bit BT.709. (ffprobe; OpenCV ignores the HDR
  transfer, as do PyAV and the ffmpeg CLI: mean saturation 26.4 / 26.6 / 26.6 on identical frames.)

## 2. What each behavior looks like (one subject, mid-video frame)

In [ ]:
subj = vt.subject.iloc[0]
for view in ["front", "side"]:
    rows = vt[(vt.subject == subj) & (vt.view == view)].sort_values("behavior_id")
    imgs = [eda.read_frame(r.path, r.n_frames // 2) for r in rows.itertuples()]
    eda.show_grid(imgs, [f"{'CHEAT' if r.label else 'normal'} | {r.behavior}" for r in rows.itertuples()])
    plt.suptitle(f"{subj} - {view}", y=1.02)

## 3. Test set format

In [ ]:
seqs = sorted(p for p in (DATA / "test").iterdir() if p.is_dir())
n_frames = pd.Series([len(list(s.glob("*.jpg"))) for s in seqs]).value_counts()
im = Image.open(seqs[0] / "01.jpg")
print("sequences:", len(seqs), "| frames per seq:", n_frames.to_dict(), "| size:", im.size)
print("luma quant table head:", list(im.quantization[0])[:8], "(= libjpeg quality 95)")
print("submission rows:", len(pd.read_csv(DATA / "sample_submission.csv")))

Test JPEGs are **640x360** (not 1280x720 as the description says), libjpeg quality 95, 4:2:0, JFIF 1.1, no ffmpeg
comment marker, i.e. OpenCV `VideoCapture -> resize -> imwrite` defaults. `dataset.prepare` reproduces exactly
this pipeline for train videos so train and test share one domain.

## 4. Test statistics on GPU vs train (brightness, saturation, motion per 1-second clip)

In [ ]:
ts = eda.test_seq_stats(DATA / "test", device=DEVICE)
display(ts.describe().round(2))

In [ ]:
sub = vt[vt.subject.isin(vt.subject.unique()[:4])]
curves = {r.video_id: eda.motion_curve(r.path) for r in sub.itertuples()}
clip_motion = pd.DataFrame([{"video_id": v, "motion": c[s:s + 29].mean()}
                            for v, c in curves.items() for s in range(0, len(c) - 29, 30)]).merge(sub, on="video_id")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
bins = np.linspace(0, max(ts.motion.quantile(0.99), clip_motion.motion.quantile(0.99)), 40)
ax[0].hist(ts.motion, bins=bins, density=True, alpha=0.5, label="test seqs")
ax[0].hist(clip_motion.motion, bins=bins, density=True, alpha=0.5, label="train 1-s clips (4 subjects)")
ax[0].set_title("Motion energy per 1-s clip")
ax[0].legend()
clip_motion.boxplot("motion", by="behavior", ax=ax[1], rot=60, fontsize=7)
ax[1].set_title("Train clip motion by behavior")
plt.suptitle("")
plt.tight_layout()

## 5. Where in the video does the behavior happen?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 5.5), sharey=True)
for ax, b in zip(axes.ravel(), BEHAVIORS, strict=True):
    for v in sub[sub.behavior == b].video_id:
        c = np.convolve(curves[v], np.ones(15) / 15, mode="same")
        ax.plot(np.arange(len(c)) / 30, c, lw=0.8, alpha=0.7)
    ax.set_title(("CHEAT: " if BEHAVIORS.index(b) >= 4 else "normal: ") + b, fontsize=9)
    ax.set_xlabel("s")
plt.tight_layout()

The label is per video, but the distinctive action (peek, look under desk, smartwatch use) is short and
episodic. Many 1-second cuts of a *cheat* video look like plain writing, so clip labels are noisy. This is the main
difficulty of the task and the reason for (a) training on many random windows per video, (b) the 8-behavior auxiliary
target, (c) careful threshold selection on out-of-fold predictions.

## 6. Test sequences are consecutive cuts of longer videos

In [ ]:
chains = eda.chain_test_sequences(DATA / "test", thr=1.2, device=DEVICE)
lens = pd.Series([len(c) for c in chains])
print(f"{len(chains)} chains cover {lens.sum()} sequences; length median {lens.median():.0f}, max {lens.max()}")
lens.value_counts().sort_index().plot.bar(figsize=(10, 3), title="Chain length (1-s sequences per reconstructed video)")

The last frame of most test sequences matches the first frame of exactly one other sequence: test = ~95 videos
(~6 unseen subjects x 16) cut into 1-second pieces and shuffled. Nearest-neighbour thumbnail distance test->train
(median 31) is at the level of *different* train subjects (29.5), far above same-subject distance (13.8): **test
subjects are not in train**, so validation must be subject-grouped (GroupKFold by subject).

## Conclusions for modelling
1. Reproduce the OpenCV 640x360 q95 pipeline for train frames (done in `prepare.py`).
2. Validate with subject-disjoint folds; report clip-level F1-macro (competition metric) plus video-level F1.
3. Clip labels are noisy (episodic actions): use 8-behavior supervision + binary marginal loss, random windows.
4. Front/side views differ strongly (HDR vs SDR colour, framing): colour jitter, crops; one model for both views.
5. Optional, rule-sensitive: aggregate predictions along reconstructed test chains (video-level context).